# Theme 5 — What Bugs Get Fixed, and Do the Fixes Hold?

Fix-specific questions (not generic PR metrics):

- **FQ1 — What *kind* of bug does each group fix?** (and which bug types get accepted)
- **FQ2 — Do the fixes actually hold?** measured by how often a merged fix is later **reverted** — including a check *within the same bug type* so it isn't just "agents fix easier bugs".

Uses PR titles (bug type) and commit messages (reverts). No file-level data needed.

In [1]:
import sys
sys.path.insert(0, '.')
from analysis_utils import (
    load_fix_prs, load_commits, merge_rate, classify_bug_type, BUG_CATEGORIES,
    set_plot_style, save_fig, AGENTS, AGENT_COLORS, THEME5_DIR,
)
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
set_plot_style()
CATS = list(BUG_CATEGORIES) + ['other']

In [2]:
# Load PRs + commits (no file-level details needed here)
df = load_fix_prs()
df['bug_type'] = df['title'].map(classify_bug_type)
agents_df = df[df['is_agent'] & df['agent'].isin(AGENTS)].copy()
human_df  = df[~df['is_agent']].copy()
commits = load_commits()
print('Agent fix PRs:', f'{len(agents_df):,}', '| Human fix PRs:', f'{len(human_df):,}')

Loading fix PRs from HuggingFace ...


  AIDev repo coverage: 8,959 distinct repos


  Survivorship cutoff at 2026-01-29: dropped 33,123 recent PRs
  Fix PRs loaded: 371,577  |  Agent: 108,080  |  Human: 263,497


Loading commits from HuggingFace ...


  Commits loaded: 1,156,238
Agent fix PRs: 108,080 | Human fix PRs: 263,497


## FQ1 — What kind of bug does each group fix?
Share of each group's fixes by bug type (from the PR title).

In [3]:
a_share = agents_df['bug_type'].value_counts(normalize=True).reindex(CATS).fillna(0)*100
h_share = human_df['bug_type'].value_counts(normalize=True).reindex(CATS).fillna(0)*100
tbl = pd.DataFrame({'Agent %':a_share.round(1), 'Human %':h_share.round(1)})
tbl['diff (A-H)'] = (tbl['Agent %']-tbl['Human %']).round(1)
print('Bug-type mix (share of each group):')
print(tbl.to_string())

# Figure: agent vs human share per bug type
order = a_share.sort_values(ascending=False).index.tolist()
x = np.arange(len(order)); w = 0.4
fig, ax = plt.subplots(figsize=(12,5))
ax.bar(x-w/2, a_share[order].values, w, label='Agent', color='#1f77b4')
ax.bar(x+w/2, h_share[order].values, w, label='Human', color='#7f7f7f')
ax.set_xticks(x); ax.set_xticklabels(order, rotation=30, ha='right')
ax.set_ylabel('% of group\'s fix PRs'); ax.set_title('FQ1: What kind of bug does each group fix?')
ax.legend()
fig.tight_layout()
save_fig(fig, 'fq1_bug_type_mix', THEME5_DIR)

Bug-type mix (share of each group):
                  Agent %  Human %  diff (A-H)
bug_type                                      
crash/error          13.3     11.3         2.0
null/undefined        0.6      0.9        -0.3
typo/format           3.2      4.3        -1.1
security              2.3      2.4        -0.1
memory/leak           0.8      1.2        -0.4
race/concurrency      1.4      1.8        -0.4
regression            0.9      2.3        -1.4
build/ci             19.4      6.0        13.4
ui/display            4.1      3.2         0.9
other                53.9     66.8       -12.9


  -> Saved: results\theme5_figures\fq1_bug_type_mix.png


WindowsPath('results/theme5_figures/fq1_bug_type_mix.png')

In [4]:
# Acceptance by bug type (agents) — is any bug type trusted less?
rows=[]
for c in CATS:
    s = agents_df[agents_df.bug_type==c]
    if len(s) >= 200:
        rows.append((c, s['is_merged'].mean()*100, len(s)))
acc = pd.DataFrame(rows, columns=['bug_type','acceptance','n']).sort_values('acceptance')
print('Agent acceptance by bug type (n>=200):')
print(acc.round(1).to_string(index=False))

fig, ax = plt.subplots(figsize=(11,5))
colors = ['#d62728' if c=='security' else '#1f77b4' for c in acc['bug_type']]
ax.barh(acc['bug_type'], acc['acceptance'], color=colors)
for i,(v,n) in enumerate(zip(acc['acceptance'], acc['n'])):
    ax.text(v+0.3, i, f'{v:.1f}% (n={n:,})', va='center', fontsize=8)
ax.set_xlabel('Agent acceptance %'); ax.set_xlim(0,100)
ax.set_title('FQ1: Agent acceptance by bug type (security in red)')
fig.tight_layout()
save_fig(fig, 'fq1_acceptance_by_bug_type', THEME5_DIR)

Agent acceptance by bug type (n>=200):
        bug_type  acceptance     n
        security        71.3  2435
  null/undefined        81.4   661
race/concurrency        81.4  1540
     memory/leak        81.5   849
     typo/format        81.7  3512
        build/ci        82.3 21002
     crash/error        83.2 14428
      ui/display        84.9  4423
           other        85.3 58223
      regression        87.7  1007


  -> Saved: results\theme5_figures\fq1_acceptance_by_bug_type.png


WindowsPath('results/theme5_figures/fq1_acceptance_by_bug_type.png')

## FQ2 — Do the fixes hold? (reverts)

A merged fix that is later **reverted** is a fix that broke something. We find revert commits (`"This reverts commit <sha>"`), map the reverted SHA back to its PR, and compute the revert rate for agent vs human merged fixes.

*Limitation: we only see revert commits that live inside a bug-fix PR, so these rates are a lower bound (applies equally to both groups).*

In [5]:
commits['message'] = commits['message'].fillna('')
rev = commits[commits['message'].str.contains('reverts commit', case=False, na=False)]
shas = rev['message'].str.extract(r'reverts commit ([0-9a-f]{7,40})', flags=re.I)[0].dropna()
reverted = set(s[:12] for s in shas)
commits['sha12'] = commits['sha'].str[:12]
reverted_prs = set(commits.loc[commits['sha12'].isin(reverted), 'pr_id'])
df['was_reverted'] = df['id'].isin(reverted_prs)

merged = df[df['is_merged']]
m_ag = merged[merged.is_agent & merged.agent.isin(AGENTS)]
m_hu = merged[~merged.is_agent]
print(f'Revert commits: {len(rev):,} | distinct reverted SHAs: {len(reverted):,}')
print(f'Agent merged fixes reverted: {m_ag.was_reverted.mean()*100:.2f}%  ({int(m_ag.was_reverted.sum()):,}/{len(m_ag):,})')
print(f'Human merged fixes reverted: {m_hu.was_reverted.mean()*100:.2f}%  ({int(m_hu.was_reverted.sum()):,}/{len(m_hu):,})')

groups = AGENTS + ['Human']
rates = [m_ag[m_ag.agent==a].was_reverted.mean()*100 for a in AGENTS] + [m_hu.was_reverted.mean()*100]
fig, ax = plt.subplots(figsize=(8,5))
ax.bar(groups, rates, color=[AGENT_COLORS[g] for g in groups])
for i,v in enumerate(rates): ax.text(i, v, f'{v:.2f}%', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('% of merged fixes later reverted'); ax.set_title('FQ2: Do the fixes hold? (revert rate)')
fig.tight_layout()
save_fig(fig, 'fq2_revert_rate', THEME5_DIR)

Revert commits: 5,508 | distinct reverted SHAs: 4,800
Agent merged fixes reverted: 0.23%  (212/90,698)
Human merged fixes reverted: 0.50%  (1,126/224,975)


  -> Saved: results\theme5_figures\fq2_revert_rate.png


WindowsPath('results/theme5_figures/fq2_revert_rate.png')

In [6]:
# Control for bug type: revert rate WITHIN each bug type (agent vs human)
rows=[]
for c in CATS:
    a = m_ag[m_ag.bug_type==c]; h = m_hu[m_hu.bug_type==c]
    if len(a)>=200 and len(h)>=200:
        rows.append((c, a.was_reverted.mean()*100, h.was_reverted.mean()*100, len(a), len(h)))
ct = pd.DataFrame(rows, columns=['bug_type','agent_rev%','human_rev%','n_ag','n_hu'])
print('Revert rate within bug type (n>=200 each):')
print(ct.round(2).to_string(index=False))

order = ct['bug_type'].tolist(); x=np.arange(len(order)); w=0.4
fig, ax = plt.subplots(figsize=(12,5))
ax.bar(x-w/2, ct['agent_rev%'], w, label='Agent', color='#1f77b4')
ax.bar(x+w/2, ct['human_rev%'], w, label='Human', color='#7f7f7f')
ax.set_xticks(x); ax.set_xticklabels(order, rotation=30, ha='right')
ax.set_ylabel('% reverted'); ax.set_title('FQ2: Revert rate within the same bug type')
ax.legend()
fig.tight_layout()
save_fig(fig, 'fq2_revert_within_bug_type', THEME5_DIR)

Revert rate within bug type (n>=200 each):
        bug_type  agent_rev%  human_rev%  n_ag   n_hu
     crash/error        0.28        0.47 12011  24924
  null/undefined        0.37        0.40   538   2014
     typo/format        0.17        0.17  2868   9498
        security        0.29        0.60  1737   5144
     memory/leak        0.14        0.40   692   2487
race/concurrency        0.16        0.71  1254   3957
      regression        0.11        0.68   883   4977
        build/ci        0.13        0.61 17276  13659
      ui/display        0.21        0.69  3755   7361
           other        0.26        0.50 49684 150954


  -> Saved: results\theme5_figures\fq2_revert_within_bug_type.png


WindowsPath('results/theme5_figures/fq2_revert_within_bug_type.png')

## Notes / caveats

- **Bug type is from PR-title keywords** — coarse, and many titles fall in 'other'. Treat shares as indicative, not exact. The large build/CI gap is robust to this noise.
- **Reverts are a lower bound** — only revert commits that occur inside a bug-fix PR are visible (same limitation for agents and humans).
- **Revert rate is also confounded by recency** — agent PRs are more recent on average, so they have had less time to be reverted. Read FQ2 alongside the within-bug-type panel.